# handlers

> Workflow-specific alignment handlers: init

In [ ]:
#| default_exp routes.handlers

In [ ]:
#| export
from typing import Any, Dict, List, Tuple, Callable, NamedTuple

from fasthtml.common import Span, APIRouter

from cjm_fasthtml_interactions.core.state_store import get_session_id
from cjm_fasthtml_card_stack.core.constants import DEFAULT_VISIBLE_COUNT, DEFAULT_CARD_WIDTH

# Local imports (page-specific, no cross-package dependencies)
from cjm_transcript_vad_align.html_ids import AlignmentHtmlIds
from cjm_transcript_vad_align.models import VADChunk, AlignmentUrls
from cjm_transcript_vad_align.services.alignment import AlignmentService

# Alignment components (same package)
from cjm_transcript_vad_align.components.step_renderer import (
    render_align_column_body, render_align_mini_stats_text,
)

# Route core helpers (same package)
from cjm_transcript_vad_align.routes.core import (
    WorkflowStateStore, _get_selection_state, _update_alignment_state,
)

# Source service for fetching media_path
from cjm_transcript_source_select.services.source import SourceService

# Debug flag for alignment data flow tracing (set False in production)
DEBUG_ALIGNMENT = True

## Initialize Handler

Fetches VAD data from the alignment service and stores in state.

In [ ]:
#| export
class AlignInitResult(NamedTuple):
    """Result from pure alignment init handler.
    
    Contains domain-specific data for the combined layer wrapper to use
    when building cross-domain OOB elements (shared chrome, alignment status).
    """
    column_body: Any  # Rendered column body content
    chunks: List[VADChunk]  # Initialized VAD chunks (all files combined)
    focused_index: int  # Focused chunk index (always 0 on init)
    visible_count: int  # Visible card count
    card_width: int  # Card stack width in rem
    media_path: str  # Path to first audio file (backward compat, may be None)
    media_paths: List[str]  # Ordered list of all audio file paths

In [ ]:
#| export
async def _handle_align_init(
    state_store:WorkflowStateStore,  # The workflow state store
    workflow_id:str,  # The workflow identifier
    source_service:SourceService,  # Service for fetching source blocks
    alignment_service:AlignmentService,  # Service for VAD analysis
    request,  # FastHTML request object
    sess,  # FastHTML session object
    urls:AlignmentUrls,  # URL bundle
    visible_count:int=DEFAULT_VISIBLE_COUNT,  # Initial visible card count
    card_width:int=DEFAULT_CARD_WIDTH,  # Initial card width in rem
) -> AlignInitResult:  # Pure domain result for wrapper to use
    """Initialize alignment from audio files via VAD plugin.
    
    Processes all selected sources' audio files, generating VAD chunks
    for each with correct audio_file_index. Returns pure domain data.
    """
    session_id = get_session_id(sess)

    if DEBUG_ALIGNMENT:
        print(f"[ALIGN_INIT] Starting alignment initialization")

    # Get selected sources from Phase 1
    selection_state = _get_selection_state(state_store, workflow_id, session_id)
    selected_sources = selection_state.get("selected_sources", [])

    if DEBUG_ALIGNMENT:
        print(f"[ALIGN_INIT] selected_sources count: {len(selected_sources)}")

    # Extract unique media_paths from all selected sources (preserving order)
    media_paths = []
    seen_paths = set()
    for source in selected_sources:
        block = source_service.get_transcription_by_id(
            record_id=source["record_id"],
            provider_id=source["provider_id"],
        )
        if block and block.media_path and block.media_path not in seen_paths:
            media_paths.append(block.media_path)
            seen_paths.add(block.media_path)

    if DEBUG_ALIGNMENT:
        print(f"[ALIGN_INIT] unique media_paths: {len(media_paths)}")
        for i, mp in enumerate(media_paths):
            print(f"[ALIGN_INIT]   [{i}] {mp}")

    # Fetch VAD data for each audio file
    all_chunks = []
    total_duration = 0.0
    global_index = 0

    if alignment_service.is_available():
        for file_idx, media_path in enumerate(media_paths):
            if DEBUG_ALIGNMENT:
                print(f"[ALIGN_INIT] Analyzing file {file_idx}: {media_path}")
            chunks, duration = await alignment_service.analyze_audio_async(media_path)
            if DEBUG_ALIGNMENT:
                print(f"[ALIGN_INIT]   -> {len(chunks)} chunks, {duration:.2f}s")
            
            # Reassign indices: global sequential + audio_file_index
            for chunk in chunks:
                chunk.index = global_index
                chunk.audio_file_index = file_idx
                global_index += 1
            
            all_chunks.extend(chunks)
            total_duration += duration

    if DEBUG_ALIGNMENT:
        print(f"[ALIGN_INIT] Total: {len(all_chunks)} chunks, {total_duration:.2f}s across {len(media_paths)} files")

    # Build audio URLs for Web Audio API
    audio_urls = []
    if urls.audio_src:
        audio_urls = [f"{urls.audio_src}?path={mp}" for mp in media_paths]

    # Serialize and store
    chunk_dicts = [c.to_dict() for c in all_chunks]
    media_path = media_paths[0] if media_paths else None
    _update_alignment_state(
        state_store, workflow_id, session_id,
        vad_chunks=chunk_dicts,
        focused_chunk_index=0,
        is_initialized=True,
        visible_count=visible_count,
        card_width=card_width,
        media_path=media_path,
        media_paths=media_paths,
        audio_duration=total_duration,
    )

    # Render column body
    column_body = render_align_column_body(
        chunks=all_chunks,
        focused_index=0,
        visible_count=visible_count,
        card_width=card_width,
        urls=urls,
        kb_system=None,
        audio_urls=audio_urls,
    )

    if DEBUG_ALIGNMENT:
        print(f"[ALIGN_INIT] Returning AlignInitResult")

    return AlignInitResult(
        column_body=column_body,
        chunks=all_chunks,
        focused_index=0,
        visible_count=visible_count,
        card_width=card_width,
        media_path=media_path,
        media_paths=media_paths,
    )

## Router Initialization

Creates the workflow router with the init route.

In [ ]:
#| export
def init_workflow_router(
    state_store:WorkflowStateStore,  # The workflow state store
    workflow_id:str,  # The workflow identifier
    source_service:SourceService,  # Service for fetching source blocks
    alignment_service:AlignmentService,  # Service for VAD analysis
    prefix:str,  # Route prefix (e.g., "/workflow/align/workflow")
    urls:AlignmentUrls,  # URL bundle (populated after routes defined)
    handler_init:Callable=None,  # Optional wrapped init handler
) -> Tuple[APIRouter, Dict[str, Callable]]:  # (router, route_dict)
    """Initialize workflow routes for alignment.
    
    Accepts optional handler override for wrapping with cross-domain
    coordination (e.g., shared chrome, alignment status OOB updates).
    """
    router = APIRouter(prefix=prefix)

    # Use provided handler or fall back to raw domain handler
    _init = handler_init or _handle_align_init

    # -------------------------------------------------------------------------
    # Workflow Operations
    # -------------------------------------------------------------------------

    @router
    async def init(request, sess):
        """Initialize alignment from audio file via VAD plugin."""
        return await _init(
            state_store, workflow_id, source_service, alignment_service,
            request, sess, urls=urls,
        )

    # -------------------------------------------------------------------------
    # Route Dict
    # -------------------------------------------------------------------------

    routes = {
        "init": init,
    }

    return router, routes

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()